In [4]:
import torch
import torch.nn as nn

# --- 1. Dataset ---
# Underlying formula: Marks = (0.4 * IQ) + (6.0 * Hours) + 5.0
X_raw = torch.tensor([
    [100.0, 2.0],  # 0.4(100) + 6.0(2) + 5.0 = 57.0
    [110.0, 4.0],  # 0.4(110) + 6.0(4) + 5.0 = 73.0
    [120.0, 3.0],  # 0.4(120) + 6.0(3) + 5.0 = 71.0
    [90.0,  6.0],  # 0.4(90)  + 6.0(6) + 5.0 = 77.0
    [105.0, 5.0]   # 0.4(105) + 6.0(5) + 5.0 = 77.0
])

y = torch.tensor([
    [57.0], 
    [73.0], 
    [71.0], 
    [77.0], 
    [77.0]
])

# --- 2. Z-Score Normalization ---
X_mean = X_raw.mean(dim=0)
X_std = X_raw.std(dim=0)
X_scaled = (X_raw - X_mean) / X_std

# --- 3. Model & Optimizer ---
model = nn.Linear(in_features=2, out_features=1)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# --- 4. Training Loop ---
for epoch in range(500):
    optimizer.zero_grad()
    predictions = model(X_scaled)
    loss = criterion(predictions, y)
    loss.backward()
    optimizer.step()

# --- 5. Batch Predictions for 5 New Test Cases ---
test_cases_raw = torch.tensor([
    [100.0, 5.0],  # Expected: 0.4(100) + 6(5) + 5 = 75.0
    [110.0, 2.0],  # Expected: 0.4(110) + 6(2) + 5 = 61.0
    [95.0,  8.0],  # Expected: 0.4(95)  + 6(8) + 5 = 91.0
    [125.0, 4.0],  # Expected: 0.4(125) + 6(4) + 5 = 79.0
    [85.0,  3.0]   # Expected: 0.4(85)  + 6(3) + 5 = 57.0
])

# Apply training normalization parameters to the batch
test_cases_scaled = (test_cases_raw - X_mean) / X_std

model.eval()
with torch.no_grad():
    predictions = model(test_cases_scaled)

# Print results comparison
print(f"{'IQ':<8} | {'Hours':<8} | {'Predicted Marks':<16} | {'Expected Marks':<15}")
print("-" * 55)

for i in range(len(test_cases_raw)):
    iq = test_cases_raw[i][0].item()
    hours = test_cases_raw[i][1].item()
    pred = predictions[i].item()
    expected = 0.4 * iq + 6.0 * hours + 5.0
    print(f"{iq:<8.1f} | {hours:<8.1f} | {pred:<16.2f} | {expected:<15.1f}")

IQ       | Hours    | Predicted Marks  | Expected Marks 
-------------------------------------------------------
100.0    | 5.0      | 75.00            | 75.0           
110.0    | 2.0      | 61.00            | 61.0           
95.0     | 8.0      | 91.00            | 91.0           
125.0    | 4.0      | 79.00            | 79.0           
85.0     | 3.0      | 57.00            | 57.0           
